In [1]:
import custom_scorer_v5
import spacy
nlp = spacy.load("./model/SMESMEoutput_FAERS_test_data_v1/model-best")
from spacy.training import Corpus
from spacy.training.example import Example

In [2]:
def eval_pred(gold, pred, case_id=None):
    M, C, S, N = 0, 0, 0, 0  # Initialize counts
    label_counts = defaultdict(lambda: {"M": 0, "C": 0, "S": 0, "N": 0})
    if True:
        # Convert entity labels to lowercase for case-insensitive comparison
        gold_ents = list(gold)
        pred_ents = list(pred)
        pred_flag = [False] * len(pred_ents)
        sorted_pred_ents = sorted(pred_ents, key=lambda x: (int(x[0]), int(x[1])))
        sorted_gold_ents = sorted(gold_ents, key=lambda x: (int(x[0]), int(x[1])))
        M_temp = 0
        C_temp = 0
        # Calculate the number of exact matches, partial matches, false positives, and false negatives
        for gold_ent in sorted_gold_ents:
            totally_match_found = False
            partial_match_found = False
            for ind1 in range(len(sorted_pred_ents)):
                pred_ent = sorted_pred_ents[ind1]
                if pred_ent[0] == gold_ent[0] and pred_ent[1] == gold_ent[1] and pred_ent[2] == gold_ent[2]:
                    totally_match_found = True
                    pred_flag[ind1] = True
                    M += 1
                    M_temp += 1
                    label_counts[gold_ent[2]]["M"] += 1
                    break
                elif pred_ent[0] == gold_ent[0] or pred_ent[1] == gold_ent[1] or gold_ent[0] < pred_ent[0] < gold_ent[1] or gold_ent[0] < pred_ent[1] < gold_ent[1] or (pred_ent[0] < gold_ent[0] and pred_ent[1] > gold_ent[0]):
                    if not pred_flag[ind1]:
                        partial_match_found = True
                        pred_flag[ind1] = True
                        C += 1
                        C_temp += 1
                        label_counts[gold_ent[2]]["C"] += 1
                        break
            if not (totally_match_found or partial_match_found):
                N += 1
                label_counts[gold_ent[2]]["N"] += 1
        S += len(sorted_pred_ents) - M_temp - C_temp
        for ind1 in range(len(sorted_pred_ents)):
            if not pred_flag[ind1]:
                label_counts[sorted_pred_ents[ind1][2]]["S"] += 1
    # Revised M', C', S', N'
    M_ = M + (0.5 * C)
    C_ = 0.5 * C
    S_ = 0.25 * S
    N_ = N
    # Compute revised Precision / Recall / F1
    precision = M_ / (M_ + C_ + S_) if (M_ + C_ + S_) > 0 else 0
    recall = M_ / (M_ + C_ + N_) if (M_ + C_ + N_) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    # Compute metrics for each label type
    label_metrics = []
    for label, counts in label_counts.items():
        M_l = counts["M"]
        C_l = counts["C"]
        S_l = counts["S"]
        N_l = counts["N"]
        M_l_ = M_l + (0.5 * C_l)
        C_l_ = 0.5 * C_l
        S_l_ = 0.25 * S_l
        N_l_ = N_l
        precision_l = M_l_ / (M_l_ + C_l_ + S_l_) if (M_l_ + C_l_ + S_l_) > 0 else 0
        recall_l = M_l_ / (M_l_ + C_l_ + N_l_) if (M_l_ + C_l_ + N_l_) > 0 else 0
        f1_l = (2 * precision_l * recall_l) / (precision_l + recall_l) if (precision_l + recall_l) > 0 else 0
        label_metrics.append({
            'case_id':case_id,
            'label':label,
            "precision": precision_l,
            "recall": recall_l,
            "f1": f1_l,
            "Matched":M_l,
            "Partially Matched":C_l,
            "Missed":N_l,
            "FP":S_l
        })
    return label_metrics


In [3]:
import pandas as pd
import os, re, json
import nltk

In [4]:
from tqdm.notebook import tqdm

In [5]:
data_folder = '../Datasets/FAERS_D1_clean'
res = []
for file in tqdm(os.listdir(data_folder)):
    if file.endswith('.json'):
        
        tmp_data = json.load(open(data_folder+'/'+file,'r'))
        text = tmp_data['pages'][0]
        annotations = tmp_data['annotations']
        human_annos = [x for x in annotations if 'SME' in x['note']]
        llm_annos = [x for x in annotations if x['note'] == 'LLM']

        for anno in human_annos:
            res.append([file, 'Human', anno['textContext']['start'], anno['textContext']['end'], anno['label'], anno['textContext']['text']])

        for anno in llm_annos:
            res.append([file, 'LLM', anno['textContext']['start'], anno['textContext']['end'], anno['label'], anno['textContext']['text']])

        curr_start = 0
        sents = re.split('↵', text)
        for sent in sents:
            # if the text parts was too long, only analyze the first available parts
            bert_res = nlp(sent[:1024])
            # if not bert_res.ents: continue
            for ent in bert_res.ents:
                start, end, content = ent.start_char+curr_start, ent.end_char+curr_start, ent.text
                if content!=text[start:end]:
                    print('Warning: ', content, text[start:end])
                    break
                res.append([file, 'BERT', start, end, ent.label_, content]) 
            curr_start += len(sent)+1


  0%|          | 0/829 [00:00<?, ?it/s]

In [7]:
total_annotation = pd.DataFrame(res, columns = ['File','Note','Start','End','Label','Text',])

In [8]:
total_annotation.to_excel('FAERS-BERT-TEST.xlsx',index=None)

In [4]:
## create a similar dataset for ETHER (for performance draw)
import pandas as pd
import os, re, json
from tqdm.notebook import tqdm

data_folder = '../Datasets/FAERS_D1_clean'
res = []
for file in tqdm(os.listdir(data_folder)):
    if file.endswith('.json'):        
        tmp_data = json.load(open(data_folder+'/'+file,'r'))
        text = tmp_data['pages'][0]
        annotations = tmp_data['annotations']
        human_annos = [x for x in annotations if 'SME' in x['note']]
        llm_annos = [x for x in annotations if x['note'] == 'LLM']
        ether_annos = [x for x in annotations if x['note'] == 'ETHER']

        for anno in human_annos:
            res.append([file, 'Human', anno['textContext']['start'], anno['textContext']['end'], anno['label'], anno['textContext']['text']])

        for anno in llm_annos:            
            res.append([file, 'LLM', anno['textContext']['start'], anno['textContext']['end'], anno['label'], anno['textContext']['text']])

        for anno in ether_annos:
            res.append([file, 'ETHER', anno['textContext']['start'], anno['textContext']['end'], anno['label'], anno['textContext']['text']])
total_annotation = pd.DataFrame(res, columns = ['File','Note','Start','End','Label','Text',])
total_annotation.to_excel('FAERS-ETHER-WHOLE.xlsx',index=None)


  0%|          | 0/829 [00:00<?, ?it/s]

In [5]:
ether_annos

[{'label': 'SYMPTOM',
  'note': 'ETHER',
  'textContext': {'page': 0,
   'start': 136,
   'end': 212,
   'text': 'concurrent conditions included smoking, arterial hypertension and depression',
   'text_raw': 'concurrent conditions included smoking, arterial hypertension and depression'},
  'relationships': {'date': {'page': 0, 'text': ''},
   'frequency': {'page': 0, 'text': ''},
   'relatives': {'page': 0, 'text': ''},
   'span': {'page': 0, 'text': ''},
   'time': {'page': 0, 'text': ''}},
  'used': 'Yes'},
 {'label': 'SYMPTOM',
  'note': 'ETHER',
  'textContext': {'page': 0,
   'start': 245,
   'end': 295,
   'text': 'patient had taken self-induced intoxication orally',
   'text_raw': 'patient taken self induced intoxication orally'},
  'relationships': {'date': {'page': 0, 'text': ''},
   'frequency': {'page': 0, 'text': ''},
   'relatives': {'page': 0, 'text': ''},
   'span': {'page': 0, 'text': ''},
   'time': {'page': 0, 'text': ''}},
  'used': 'Yes'},
 {'label': 'DRUG',
  'note